# 🏥 VitalSync: ML Proof of Concept — 25% Milestone
## Predictive Patient Risk Classification & AI Clinical Summarization

**Project**: VitalSync — Decoupled Microservices API for Healthcare Data Ingestion & NLP
**Domain**: Internet of Medical Things (IoMT), Generative AI
**SDG Alignment**: UN SDG 3 — Good Health and Well-being
**Dataset**: Human Vital Sign Dataset (Kaggle — nasirayub2)

### What This Notebook Proves:
1. ✅ A trained ML model can **predict patient risk level** from vital signs
2. ✅ An AI can **convert raw numbers into clinical summaries** for doctors
3. ✅ The data pipeline from raw telemetry → actionable insight is feasible

In [ ]:
!pip install -q kagglehub pandas numpy matplotlib seaborn scikit-learn xgboost google-generativeai

In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, 
    accuracy_score, ConfusionMatrixDisplay, roc_curve, auc
)
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

# Set plot style
plt.style.use('dark_background')
sns.set_palette("viridis")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

In [ ]:
# Download the dataset
path = kagglehub.dataset_download("nasirayub2/human-vital-sign-dataset")
print("Path to dataset files:", path)

# Load into pandas
import os
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
df = pd.read_csv(os.path.join(path, csv_files[0]))

print(f"\n📊 Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"📋 Columns: {list(df.columns)}")
display(df.head(10))

## 📊 Section 2: Exploratory Data Analysis
Understanding the dataset before building our model.

In [ ]:
# Basic statistics
print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)
print(f"Total Records: {len(df):,}")
print(f"Missing Values: {df.isnull().sum().sum()}")
print(f"Duplicate Rows: {df.duplicated().sum()}")
print(f"\nData Types:\n{df.dtypes}")
print(f"\nStatistical Summary:")
display(df.describe().round(2))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('🏥 Raw Patient Vital Signs — Information Overload for Clinicians', 
             fontsize=16, fontweight='bold', color='#60a5fa')

vitals = ['Heart Rate', 'Oxygen Saturation', 'Systolic Blood Pressure', 'Diastolic Blood Pressure', 'Body Temperature', 'Respiratory Rate']

for ax, vital in zip(axes.flat, vitals):
    if vital in df.columns:
        ax.plot(df[vital].head(200), linewidth=0.5, alpha=0.7, color='#f472b6')
        ax.set_title(vital, fontsize=12, color='white')
        ax.set_xlabel('Reading #', fontsize=9)
        ax.tick_params(colors='gray')

plt.tight_layout()
plt.savefig('plot1_raw_data_chaos.png', dpi=150, bbox_inches='tight', facecolor='#0a0e1a')
plt.show()
print("👆 This is what a doctor sees: hundreds of fluctuating numbers. VitalSync turns this into 3 readable sentences.")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Bar chart
risk_counts = df['Risk Category'].value_counts()
colors = {'Low Risk': '#10b981', 'Normal': '#3b82f6', 'High Risk': '#ef4444'}
ax1.bar(risk_counts.index, risk_counts.values, color=[colors.get(x, '#6b7280') for x in risk_counts.index])
ax1.set_title('Patient Risk Distribution', fontsize=14, fontweight='bold')
ax1.set_ylabel('Number of Patients')

# Pie chart
ax2.pie(risk_counts.values, labels=risk_counts.index, autopct='%1.1f%%',
        colors=[colors.get(x, '#6b7280') for x in risk_counts.index],
        textprops={'color': 'white'})
ax2.set_title('Risk Level Breakdown', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('plot2_risk_distribution.png', dpi=150, bbox_inches='tight', facecolor='#0a0e1a')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))

color_map = {'Low Risk': '#10b981', 'Normal': '#3b82f6', 'High Risk': '#ef4444'}
for risk, color in color_map.items():
    mask = df['Risk Category'] == risk
    # Subsample to avoid overlapping 200k points
    subset = df.loc[mask].sample(min(2000, len(df.loc[mask])), random_state=42) 
    ax.scatter(subset['Heart Rate'], subset['Oxygen Saturation'],
               c=color, label=risk, alpha=0.6, s=15, edgecolors='none')

# Add danger zone shading
ax.axhline(y=92, color='#ef4444', linestyle='--', alpha=0.5, label='SpO2 Critical Threshold (92%)')
ax.axvline(x=100, color='#f59e0b', linestyle='--', alpha=0.5, label='HR Warning Threshold (100 bpm)')

ax.set_xlabel('Heart Rate (bpm)', fontsize=13)
ax.set_ylabel('SpO2 — Blood Oxygen (%)', fontsize=13)
ax.set_title('🫀 Patient Health Risk Spectrum\nHeart Rate vs Blood Oxygen Saturation', 
             fontsize=15, fontweight='bold', color='#60a5fa')
ax.legend(loc='lower left', fontsize=10)

plt.savefig('plot3_risk_spectrum.png', dpi=150, bbox_inches='tight', facecolor='#0a0e1a')
plt.show()
print("👆 Green = Safe patients. Red = Critical patients. The AI focuses on the red zone.")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

# Drop non-numeric and derived columns that duplicate info for the matrix
cols_to_corr = ['Heart Rate', 'Oxygen Saturation', 'Systolic Blood Pressure', 
                'Diastolic Blood Pressure', 'Body Temperature', 'Respiratory Rate', 'Age', 'Weight (kg)']
corr_matrix = df[cols_to_corr].corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Correlation Coefficient'})

ax.set_title('Feature Correlation Matrix — Vital Signs Relationships', 
             fontsize=14, fontweight='bold', color='#60a5fa')

plt.savefig('plot4_correlation_heatmap.png', dpi=150, bbox_inches='tight', facecolor='#0a0e1a')
plt.show()
print("👆 Shows which vital signs are statistically related.")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Vital Sign Distributions by Risk Level', fontsize=16, fontweight='bold', color='#60a5fa')

for ax, vital in zip(axes.flat, vitals):
    if vital in df.columns:
        sns.boxplot(data=df, x='Risk Category', y=vital, ax=ax,
                    palette={'Low Risk': '#10b981', 'Normal': '#3b82f6', 'High Risk': '#ef4444'})
        ax.set_title(vital, fontsize=12)

plt.tight_layout()
plt.savefig('plot5_boxplots_by_risk.png', dpi=150, bbox_inches='tight', facecolor='#0a0e1a')
plt.show()

## 🔧 Section 3: Feature Engineering
Creating derived clinical features that improve model accuracy.

In [ ]:
# The dataset actually already includes some derived features (HRV, Pulse Pressure, BMI, MAP)
# Let's verify them and optionally add Shock Index

df['Shock_Index'] = df['Heart Rate'] / df['Systolic Blood Pressure']

print(f"✅ Features ready. Total columns: {df.shape[1]}")
display(df[['Derived_Pulse_Pressure', 'Derived_MAP', 'Shock_Index']].describe().round(3))

In [ ]:
# Encode target variable
le = LabelEncoder()
df['Risk_Encoded'] = le.fit_transform(df['Risk Category'])
print(f"Class mapping: {dict(zip(le.classes_, range(len(le.classes_))))}")

# Select features (drop non-numeric, ID, timestamp, and target)
drop_cols = ['Patient ID', 'Timestamp', 'Risk Category', 'Risk_Encoded']
feature_cols = [c for c in df.columns if c not in drop_cols and np.issubdtype(df[c].dtype, np.number)]

X = df[feature_cols]
y = df['Risk_Encoded']

# For PoC, let's sample 50,000 rows to train fast, but stratify it
X_sampled, _, y_sampled, _ = train_test_split(X, y, train_size=50000, stratify=y, random_state=42)

# Train/Test split (80/20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X_sampled, y_sampled, test_size=0.2, random_state=42, stratify=y_sampled
)

print(f"\n📊 Train set: {X_train.shape[0]:,} samples")
print(f"📊 Test set:  {X_test.shape[0]:,} samples")
print(f"📊 Features:  {X_train.shape[1]} columns")

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Features scaled using StandardScaler (zero mean, unit variance)")

## 🌲 Section 4: Model Training & Evaluation

In [ ]:
import time
start = time.time()

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)
rf_model.fit(X_train_scaled, y_train)
rf_pred = rf_model.predict(X_test_scaled)
rf_accuracy = accuracy_score(y_test, rf_pred)

ml_train_time = time.time() - start
print(f"🌲 Random Forest Accuracy: {rf_accuracy:.4f} ({rf_accuracy*100:.1f}%)")
print(f"⏱️ Training time: {ml_train_time:.2f} seconds")

In [ ]:
start = time.time()
gb_model = GradientBoostingClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42
)
gb_model.fit(X_train_scaled, y_train)
gb_pred = gb_model.predict(X_test_scaled)
gb_accuracy = accuracy_score(y_test, gb_pred)

print(f"🚀 Gradient Boosting Accuracy: {gb_accuracy:.4f} ({gb_accuracy*100:.1f}%)")
print(f"⏱️ Training time: {time.time() - start:.2f} seconds")

# Pick best model
best_model = rf_model if rf_accuracy >= gb_accuracy else gb_model
best_pred = rf_pred if rf_accuracy >= gb_accuracy else gb_pred
best_name = "Random Forest" if rf_accuracy >= gb_accuracy else "Gradient Boosting"
print(f"\n✅ Best Model: {best_name}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Confusion Matrix
cm = confusion_matrix(y_test, best_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=le.classes_)
disp.plot(ax=ax1, cmap='Blues', colorbar=False)
ax1.set_title(f'Confusion Matrix — {best_name}', fontsize=14, fontweight='bold')
ax1.grid(False)

# Classification Report Heatmap
report = classification_report(y_test, best_pred, target_names=le.classes_, output_dict=True)
report_df = pd.DataFrame(report).iloc[:3, :len(le.classes_)].T
sns.heatmap(report_df, annot=True, fmt='.3f', cmap='Greens', ax=ax2,
            linewidths=0.5, vmin=0.5, vmax=1.0)
ax2.set_title('Classification Metrics per Risk Level', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('plot6_confusion_matrix.png', dpi=150, bbox_inches='tight', facecolor='#0a0e1a')
plt.show()

print("\n📋 Full Classification Report:")
print(classification_report(y_test, best_pred, target_names=le.classes_))

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    feat_imp = pd.Series(importances, index=feature_cols).sort_values(ascending=True)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(feat_imp)))
    feat_imp.plot(kind='barh', ax=ax, color=colors)
    ax.set_title(f'Feature Importance — {best_name}\nWhich vital signs matter most?',
                 fontsize=14, fontweight='bold', color='#60a5fa')
    ax.set_xlabel('Importance Score')
    
    plt.savefig('plot7_feature_importance.png', dpi=150, bbox_inches='tight', facecolor='#0a0e1a')
    plt.show()

## 🤖 Section 5: AI Clinical Summarization (NLP)
Now we connect the ML model's prediction to an LLM that writes human-readable clinical summaries.

**Pipeline**: Raw Vital Signs → ML Model (Risk Prediction) → LLM (Clinical Summary)

In [ ]:
import google.generativeai as genai

# PUT YOUR API KEY HERE
GEMINI_API_KEY = "PUT_API_KEY_HERE"
genai.configure(api_key=GEMINI_API_KEY)

model = genai.GenerativeModel('gemini-2.0-flash')
print("✅ Gemini 2.0 Flash configured")

In [ ]:
SYSTEM_PROMPT = """You are VitalSync-AI, an expert clinical decision support assistant.
Analyze the patient vital sign data below and generate a structured 3-bullet clinical summary.

STRICT RULES:
1. Use ONLY the provided numerical data. Do NOT invent symptoms.
2. Use standard medical terminology (sinus tachycardia, arterial oxygen desaturation, etc.).
3. Output EXACTLY 3 bullets — no greetings, no markdown formatting, no closings.

BULLET FORMAT:
• Bullet 1 (Hemodynamic Status): Stability of HR, BP, and SpO2 trends.
• Bullet 2 (Critical Deviations): Any vitals exceeding clinical thresholds.
• Bullet 3 (Recommended Action): Immediate bedside action for the attending physician.
"""

def build_prompt(row, risk_prediction):
    prompt = f"""Patient Telemetry Data:
- Heart Rate: {row.get('Heart Rate', 'N/A')} bpm
- SpO2 (Oxygen Saturation): {row.get('Oxygen Saturation', 'N/A')}%
- Blood Pressure: {row.get('Systolic Blood Pressure', 'N/A')}/{row.get('Diastolic Blood Pressure', 'N/A')} mmHg
- Body Temperature: {row.get('Body Temperature', 'N/A')} °C
- Respiratory Rate: {row.get('Respiratory Rate', 'N/A')} breaths/min
- ML Model Risk Prediction: {risk_prediction}

Generate the 3-bullet clinical summary now:"""
    return prompt

In [ ]:
# Select 3 diverse cases from the test set
test_indices = []
for risk_class in range(len(le.classes_)):
    idx = y_test[y_test == risk_class].index[0]
    test_indices.append(idx)

print("=" * 80)
print("🏥 VitalSync AI — BEFORE & AFTER DEMONSTRATION")
print("=" * 80)

severity_emoji = {'Low Risk': '🟢', 'Normal': '🔵', 'High Risk': '🔴'}

for i, idx in enumerate(test_indices):
    row = df.loc[idx]
    predicted_class = le.classes_[best_model.predict(scaler.transform(X.loc[[idx]]))[0]]
    
    print(f"\n{'─' * 80}")
    print(f"PATIENT CASE {i+1}  |  ML Prediction: {severity_emoji.get(predicted_class, '⚪')} {predicted_class}")
    print(f"{'─' * 80}")
    
    # BEFORE: Raw data
    print(f"\n📊 BEFORE (What the doctor sees — raw numbers):")
    print(f"   HR: {row.get('Heart Rate', 'N/A')} | SpO2: {row.get('Oxygen Saturation', 'N/A')}% | "
          f"BP: {row.get('Systolic Blood Pressure', 'N/A')}/{row.get('Diastolic Blood Pressure', 'N/A')} | "
          f"Temp: {row.get('Body Temperature', 'N/A')}°C | RR: {row.get('Respiratory Rate', 'N/A')}")
    
    # AFTER: AI Summary
    prompt = build_prompt(row.to_dict(), predicted_class)
    
    if GEMINI_API_KEY != "PUT_API_KEY_HERE":
        try:
            response = model.generate_content(SYSTEM_PROMPT + "\n\n" + prompt)
            print(f"\n🤖 AFTER (What VitalSync shows the doctor — AI summary):")
            print(f"   {response.text.strip()}")
        except Exception as e:
            print(f"\n⚠️ API Error (Using Fallback): {e}")
            print(f"   • Hemodynamic Status: HR {row['Heart Rate']} bpm, BP {row['Systolic Blood Pressure']}/{row['Diastolic Blood Pressure']} mmHg.")
            print(f"   • Critical Deviations: Risk level predicted as {predicted_class}.")
            print(f"   • Recommended Action: Automated summary offline. Review manually.")
    else:
        print(f"\n🤖 AFTER (MOCK FALLBACK — API KEY NOT SET):")
        print(f"   • Hemodynamic Status: HR {row['Heart Rate']} bpm, BP {row['Systolic Blood Pressure']}/{row['Diastolic Blood Pressure']} mmHg.")
        print(f"   • Critical Deviations: Risk level predicted as {predicted_class}.")
        print(f"   • Recommended Action: Assess patient based on raw vitals.")

print(f"\n{'=' * 80}")
print("👆 VitalSync converts confusing numbers into actionable medical advice.")
print(f"{'=' * 80}")

In [ ]:
# Benchmark ML model inference
start = time.time()
for _ in range(1000):
    best_model.predict(X_test_scaled[:1])
ml_time = (time.time() - start) / 1000 * 1000  # ms per prediction

print(f"⚡ ML Model (Risk Classification) Latency: {ml_time:.2f} ms per prediction")